# 12 — Transformer Blocks and GPT

## Goal

Previous lessons studied the main components of a decoder-only
Transformer independently:

- token embeddings,
- causal self-attention,
- multi-head attention,
- RMSNorm,
- residual connections,
- feed-forward networks,
- SwiGLU,
- and Rotary Position Embeddings.

This lesson assembles those components into a complete decoder-only
language model.

The main objective is to understand how information flows through the
entire model:

$$
\text{token IDs}
\rightarrow
\text{embeddings}
\rightarrow
\text{Transformer blocks}
\rightarrow
\text{final normalization}
\rightarrow
\text{language-model head}
\rightarrow
\text{logits}.
$$

The focus is no longer on reimplementing mechanisms that have already
been studied.

Instead, the focus is on **composition**:

- how modules connect,
- which tensor shapes must be preserved,
- where RoPE is applied,
- how residual streams flow through depth,
- and how the final hidden states become next-token logits.

In [1]:
from dataclasses import dataclass
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

## 1. The Decoder-Only Transformer

A decoder-only Transformer maintains a residual stream with shape

$$
X \in \mathbb{R}^{B \times T \times C}.
$$

The model begins with token IDs:

$$
(B,T),
$$

which are converted into token embeddings:

$$
(B,T)
\rightarrow
(B,T,C).
$$

The representation then passes through a stack of Transformer blocks.

Each block preserves the same residual-stream shape:

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

After the final block, a normalization layer is applied, followed by a
language-model head:

$$
(B,T,C)
\rightarrow
(B,T,V),
$$

where $V$ is the vocabulary size.

The output tensor contains one vocabulary-logit vector for every token
position.

![Transformer Block](./imgs/transformer_block.png)


In [ ]:
from typing import Any


class TransformerBlock(nn.Module):
    """A pre-norm decoder-only Transformer block.

    The block contains two residual sublayers:

    1. RMSNorm followed by causal self-attention.
    2. RMSNorm followed by a feed-forward network.

    Both sublayers preserve the residual-stream shape `(B, T, C)`.

    Args:
        embedding_dim: Width of the residual stream.
        attention: Causal self-attention module.
        feed_forward: Token-wise feed-forward module."""

    def __init__(
        self, embedding_dim: int, attention: nn.Module, feed_forward: nn.Module
    ) -> None:
        super().__init__()
        self.attention_norm = nn.RMSNorm(embedding_dim)
        self.attention = attention
        self.feed_forward_norm = nn.RMSNorm(embedding_dim)

        self.feed_forward = feed_forward

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply attention and feed-forward residual updates.

        Args:
            x: Residual-stream tensor of shape `(B, T, C)`.

        Returns:
            Updated residual-stream tensor with shape `(B, T, C)`.
        """
        # The original residual stream bypasses the attention sublayer.
        x = self.attention(self.attention_norm(x))
        # The updated residual stream then bypasses the MLP sublayer.
        x = x + self.feed_forward(self.feed_forward_norm(x))

        return x


## 2. RoPE-Aware Causal Self-Attention

Previous lessons implemented causal attention and RoPE separately.

We now combine them into a reusable attention module.

The input is the residual-stream representation

$$
X \in \mathbb{R}^{B \times T \times C}.
$$

Query, key, and value projections preserve the total model width:

$$
Q,K,V
\in
\mathbb{R}^{B \times T \times C}.
$$

They are then split into $H$ attention heads:

$$
(B,T,C)
\rightarrow
(B,H,T,D),
$$

where

$$
D=\frac{C}{H}.
$$

RoPE is applied to queries and keys before the attention scores are
computed.

The attention output is finally concatenated back into

$$
(B,T,C)
$$

and projected into the residual stream.

In [2]:
def rope_frequencies(
    head_dim: int,
    base: float = 10000.0,
    *,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Create the angular frequencies used by Rotary Position Embeddings.

    Args:
        head_dim: Feature dimension of one attention head.
        base: Base controlling the geometric spacing of frequencies.
        device: Device on which to create the frequency tensor.

    Returns:
        A tensor of shape `(head_dim / 2,)`, containing one angular
        frequency for each adjacent feature pair.

    Raises:
        ValueError: If `head_dim` is not even.
    """
    if head_dim % 2 != 0:
        raise ValueError("head_dim must be even for RoPE.")

    # Every adjacent feature pair shares one rotation frequency.
    dimension_indices: torch.Tensor = torch.arange(
        0,
        head_dim,
        2,
        dtype=torch.float32,
        device=device,
    )

    return torch.pow(
        base,
        -dimension_indices / head_dim,
    )

In [3]:
def build_rope_cache(
    sequence_length: int,
    head_dim: int,
    *,
    base: float = 10000.0,
    device: torch.device | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Precompute cosine and sine values used by RoPE.

    Args:
        sequence_length: Number of token positions.
        head_dim: Feature dimension of one attention head.
        base: Base controlling the range of RoPE frequencies.
        device: Device on which to create the tensors.

    Returns:
        A pair `(cos_values, sin_values)`, each with shape
        `(sequence_length, head_dim / 2)`.
    """
    frequencies: torch.Tensor = rope_frequencies(
        head_dim=head_dim,
        base=base,
        device=device,
    )

    positions: torch.Tensor = torch.arange(
        sequence_length,
        dtype=torch.float32,
        device=device,
    )

    # Create one rotation angle for every (position, frequency) pair.
    angles: torch.Tensor = positions.unsqueeze(1) * frequencies.unsqueeze(0)

    return (
        torch.cos(angles),
        torch.sin(angles),
    )

In [4]:
def apply_rope(
    x: torch.Tensor,
    cos_values: torch.Tensor,
    sin_values: torch.Tensor,
) -> torch.Tensor:
    """Apply RoPE to an attention tensor.

    Args:
        x: Tensor of shape `(B, H, T, D)`.
        cos_values: Cosine values of shape `(T, D / 2)`.
        sin_values: Sine values of shape `(T, D / 2)`.

    Returns:
        Rotated tensor with the same shape `(B, H, T, D)`.
    """
    x_even: torch.Tensor = x[..., 0::2]
    x_odd: torch.Tensor = x[..., 1::2]

    # Add batch and head axes so the same positional rotations
    # broadcast across all examples and attention heads.
    cos_values = cos_values.unsqueeze(0).unsqueeze(0)
    sin_values = sin_values.unsqueeze(0).unsqueeze(0)

    rotated_even: torch.Tensor = x_even * cos_values - x_odd * sin_values

    rotated_odd: torch.Tensor = x_even * sin_values + x_odd * cos_values

    # Reconstruct adjacent rotated feature pairs.
    rotated_pairs: torch.Tensor = torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    )

    return rotated_pairs.flatten(start_dim=-2)

In [9]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with Rotary Position Embeddings.

    Args:
        embedding_dim: Width of the residual stream.
        num_heads: Number of attention heads.
        rope_base: Base used to construct RoPE frequencies.
    """

    def __init__(
        self, embedding_dim: int, num_heads: int, rope_base: float = 10000.0
    ) -> None:
        super().__init__()
        if embedding_dim % num_heads != 0:
            raise ValueError("embedding_dim mut be divisible by num_heads")

        self.embedding_dim: int = embedding_dim
        self.num_heads: int = num_heads
        self.head_num: int = embedding_dim // num_heads
        self.rope_base: float = rope_base

        if self.head_num % 2 != 0:
            raise ValueError("head_dim must be even for RoPE")

        self.query_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )
        self.key_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )
        self.value_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )
        self.output_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply RoPE-aware causal self-attention.

        Args:
            x: Residual-stream tensor of shape `(B, T, C)`.

        Returns:
            Attention output with shape `(B, T, C)`.
        """
        _, sequence_length, _ = x.shape

        q: torch.Tensor = self.query_projection(x)
        k: torch.Tensor = self.key_projection(x)
        v: torch.Tensor = self.value_projection(x)

        # split heads
        q = rearrange(q, "b t (h d) -> b h t d", h=self.num_heads)
        k = rearrange(k, "b t (h d) -> b h t d", h=self.num_heads)
        v = rearrange(v, "b t (h d) -> b h t d", h=self.num_heads)

        # build RoPE cache
        cos_values, sin_values = build_rope_cache(
            sequence_length=sequence_length,
            head_dim=self.head_num,
            base=self.rope_base,
            device=x.device,
        )

        # apply RoPE only to Q and K
        q = apply_rope(q, cos_values, sin_values)
        k = apply_rope(k, cos_values, sin_values)

        # attention
        # scores = q @ k.transpose(-2, -1)
        # scores /= torch.sqrt(d)
        # mask -> softmax -> weights @ V -> output

        attended: torch.Tensor = F.scaled_dot_product_attention(
            q, k, v, is_causal=True
        )
        # Concatenate all attention heads back into model width C.
        attended = rearrange(
            attended,
            "b h t d -> b t (h d)",
        )

        return self.output_projection(attended)

### Using an Optimized Attention Primitive

Earlier lessons implemented scaled dot-product attention explicitly.

At this stage, the mechanism is already understood, so the model uses

```py
F.scaled_dot_product_attention(...)
```

as the attention primitive.

This preserves the learning principle:

implement a mechanism explicitly when it is new; use a mature
implementation after its tensor semantics and mathematics are
understood.

The surrounding architecture — projections, head organization, RoPE,
residual structure, and block composition — remains explicit.


---

# 10. Concatenate heads

```py
        attended = rearrange(
            attended,
            "b h t d -> b t (h d)",
        )
```

In [10]:
# shape test
attention = CausalSelfAttention(
    embedding_dim=32,
    num_heads=4,
)

x: torch.Tensor = torch.randn(
    2,
    10,
    32,
)

output: torch.Tensor = attention(x)

print("input:", x.shape)
print("output:", output.shape)

input: torch.Size([2, 10, 32])
output: torch.Size([2, 10, 32])


In [11]:
# causality verification
attention.eval()

x_original: torch.Tensor = torch.randn(
    1,
    6,
    32,
)

x_modified: torch.Tensor = x_original.clone()

# Change only future tokens.
x_modified[:, 4:, :] = torch.randn_like(x_modified[:, 4:, :])

with torch.no_grad():
    output_original: torch.Tensor = attention(x_original)

    output_modified: torch.Tensor = attention(x_modified)

print(
    "Earlier positions unchanged:",
    torch.allclose(
        output_original[:, :4, :],
        output_modified[:, :4, :],
        atol=1e-5,
    ),
)

Earlier positions unchanged: True
